## Preprocessing and DataLoader

In [22]:
from torchvision import datasets, transforms
from torch.utils.data import  DataLoader

In [23]:
# Transformation des images
transform = transforms.Compose([
    transforms.Resize((128,128)), #redimensionner
    transforms.ToTensor (), # convertir en tensor
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]) # normaliser
])

In [24]:
# Charger les données
train_data = datasets.ImageFolder("data/training_set", transform=transform)
test_data = datasets.ImageFolder("data/test_set", transform=transform)

In [25]:
# Créer les DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

## Construire le CNN

In [26]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear( 64*32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

## Entraînement

In [27]:
import torch

model = CNN()
criterion = nn.CrossEntropyLoss()           # loss adaptée à classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Boucle d'entraînement
for epoch in range(5):  # nombre d'époques
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()               # reset gradients
        outputs = model(images)             # forward
        loss = criterion(outputs, labels)   # calcul loss
        loss.backward()                     # backpropagation
        optimizer.step()                    # update paramètres
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.6090
Epoch 2, Loss: 0.5071
Epoch 3, Loss: 0.8904
Epoch 4, Loss: 0.6948
Epoch 5, Loss: 0.3565


## Validation

In [29]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)   # predicted shape: (batch_size,)
        total += labels.size(0)                # nombre d’images dans ce batch
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")


Validation Accuracy: 73.55%


## Inference

In [34]:
from PIL import Image

img = Image.open("cheval.jpeg")
img = transform(img).unsqueeze(0)

model.eval()
with torch.no_grad():
    output = model(img)
    pred = torch.argmax(output, dim=1)
    print("Cat" if pred.item() == 0 else "Dog")

Dog


## Sauvergarde du modèle


In [35]:
torch.save(model.state_dict(), "models/cnn_cats_dogs.pth")
print("Done")

Done
